In [ ]:
import torch
import torch.optim as optim
import gymnasium
import numpy as np

from RLAlg.buffer.replay_buffer import ReplayBuffer
from RLAlg.alg.ddpg import DDPG
from model import DDPGActor, DDPGCritic

In [ ]:
def setup_env(env_name, mode=None):
    env = gymnasium.make(env_name, render_mode=mode)
    env = gymnasium.wrappers.RecordEpisodeStatistics(env)

    return env

In [ ]:
env_name = "Pendulum-v1"
env_num = 10
max_steps = 200

envs = gymnasium.vector.SyncVectorEnv([lambda: setup_env(env_name) for _ in range(env_num)])

obs_space = envs.single_observation_space.shape
action_space = envs.single_action_space.shape
max_action = envs.single_action_space.high[0]

actor = DDPGActor(np.prod(obs_space), np.prod(action_space), [128, 128], max_action)
critic = DDPGCritic(np.prod(obs_space), np.prod(action_space), [128, 128])
actor_target = DDPGActor(np.prod(obs_space), np.prod(action_space), [128, 128])
critic_target = DDPGCritic(np.prod(obs_space), np.prod(action_space), [128, 128])

actor_target.load_state_dict(actor.state_dict())
critic_target.load_state_dict(critic.state_dict())

for param in actor_target.parameters():
    param.requires_grad = False
    
for param in critic_target.parameters():
    param.requires_grad = False
    

actor_optimizer = optim.Adam(actor.parameters(), lr=3e-4)
critic_optimizer = optim.Adam(critic.parameters(), lr=3e-4)

buffer = ReplayBuffer(env_num, int(1e6), obs_space, action_space)

In [ ]:
def average_non_zero(numbers):
    non_zero_numbers = [num for num in numbers if num != 0]
    if not non_zero_numbers:
        return 0  # Return 0 if there are no non-zero elements
    return sum(non_zero_numbers) / len(non_zero_numbers)

In [ ]:
@torch.no_grad()
def get_action(obs, deterministic=False):
    obs = torch.as_tensor(obs).float()
    action = actor(obs)
    if not deterministic:
        noise = torch.randn_like(action) * 0.1
        action += noise
        
    action = torch.clamp(action, -1, 1)
    
    return action.tolist()

In [ ]:
def rollout():
    obs, info = envs.reset()
    for i in range(200):
        action = get_action(obs, False)
        next_obs, reward, done, timeout, info = envs.step(action)
        buffer.add_steps(obs, action, reward, done, next_obs)
        obs = next_obs
        
        if "episode" in info:
            print(average_non_zero(info['episode']['r']))
            

In [ ]:
def update():
    for _ in range(50):
        batch = buffer.sample(200)
        obsx = batch["states"]
        action = batch["actions"]
        reward = batch["rewards"]
        done = batch["dones"]
        next_obs = batch["next_states"]
        
        critic_loss = DDPG.compute_critic_loss(actor_target, critic, critic_target, obsx, action, reward, next_obs, done, 0.99)
        critic_optimizer.zero_grad()
        critic_loss.backward()
        critic_optimizer.step()
        
        for param in critic.parameters():
            param.requires_grad = False
        
        actor_loss = DDPG.compute_actor_loss(actor, critic, obsx)
        actor_optimizer.zero_grad()
        actor_loss.backward()
        actor_optimizer.step()
        
        for param in critic.parameters():
            param.requires_grad = True
            
        DDPG.update_target_param(actor, actor_target, 0.005)
        DDPG.update_target_param(critic, critic_target, 0.005)

In [ ]:
for _ in range(1000):
    rollout()
    update()